##### Q5 - Single Trusted Inventory View
##### Q6 - Identify warehouses/SKUs that may need attention

##### Create analytical views for management

In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from typing import Dict, Tuple, Optional
import json
from datetime import datetime, timedelta

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [3]:
def load_integrated_data(integrated_dir: Path) -> pd.DataFrame:
    """
    Load the master integrated data
    
    Args:
        integrated_dir: Path to integrated data directory
    
    Returns:
        Integrated dataframe
    """
    logger.info(f"Loading integrated data from: {integrated_dir}")
    
    filepath = integrated_dir / 'master_integrated_data.csv'
    
    if filepath.exists():
        df = pd.read_csv(filepath)
        logger.info(f"  - Loaded integrated data: {len(df)} rows, {len(df.columns)} columns")
        return df
    else:
        logger.error(f"  - Integrated data not found: {filepath}")
        return None

In [4]:
def create_inventory_view(integrated_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a single trusted inventory view at SKU + Warehouse level
    """
    logger.info("\n" + "="*60)
    logger.info("CREATING INVENTORY VIEW (Q5)")
    logger.info("="*60)
    
    # Make a copy
    df = integrated_df.copy()
    
    # ==========================================
    # Step 1: Identify key columns
    # ==========================================
    
    # Find date column
    date_col = None
    for col in df.columns:
        if col in ['date', 'snapshot_date', 'order_date', 'transaction_date']:
            date_col = col
            break
    
    if date_col is None:
        for col in df.columns:
            if 'date' in col.lower():
                date_col = col
                break
    
    # Find quantity columns
    sales_qty_col = None
    for col in df.columns:
        if 'sales_quantity' in col.lower() or 'sales qty' in col.lower():
            sales_qty_col = col
            break
    
    orders_qty_col = None
    for col in df.columns:
        if 'ordered_quantity' in col.lower() or 'ordered qty' in col.lower():
            orders_qty_col = col
            break
    
    inv_col = None
    for col in df.columns:
        if 'on_hand' in col.lower():
            inv_col = col
            break
    
    # Find revenue column
    revenue_col = None
    for col in df.columns:
        if 'revenue' in col.lower():
            revenue_col = col
            break
    
    logger.info(f"  Using columns:")
    logger.info(f"    Date: {date_col}")
    logger.info(f"    Sales Qty: {sales_qty_col}")
    logger.info(f"    Orders Qty: {orders_qty_col}")
    logger.info(f"    Inventory: {inv_col}")
    logger.info(f"    Revenue: {revenue_col}")
    
    # ==========================================
    # Step 2: Convert dates and calculate periods
    # ==========================================
    
    if date_col and date_col in df.columns:
        df['date_parsed'] = pd.to_datetime(df[date_col], errors='coerce')
        df['days_ago'] = (pd.Timestamp.now() - df['date_parsed']).dt.days
    else:
        logger.warning("  No valid date column found - using current date")
        df['days_ago'] = 0
    
    # ==========================================
    # Step 3: Aggregate at SKU + Warehouse level
    # ==========================================
    
    # Ensure we have SKU and Warehouse columns
    sku_col = None
    wh_col = None
    
    for col in df.columns:
        if col.lower() == 'sku':
            sku_col = col
        if col.lower() == 'warehouse_id':
            wh_col = col
    
    if sku_col is None:
        for col in df.columns:
            if 'sku' in col.lower():
                sku_col = col
                break
    
    if wh_col is None:
        for col in df.columns:
            if 'warehouse' in col.lower() and 'id' in col.lower():
                wh_col = col
                break
    
    if sku_col is None or wh_col is None:
        logger.error("  Could not identify SKU and Warehouse columns!")
        return None
    
    logger.info(f"  Aggregating by: {sku_col} + {wh_col}")
    
    # Prepare aggregation dictionary
    agg_dict = {}
    
    if inv_col:
        agg_dict[inv_col] = 'mean'
    if sales_qty_col:
        agg_dict[sales_qty_col] = 'sum'
    if orders_qty_col:
        agg_dict[orders_qty_col] = 'sum'
    if revenue_col:
        agg_dict[revenue_col] = 'sum'
    if 'transaction_id' in df.columns:
        agg_dict['transaction_id'] = 'nunique'
    if 'order_id' in df.columns:
        agg_dict['order_id'] = 'nunique'
    if date_col:
        agg_dict[date_col] = 'max'
    
    # Create view at SKU + Warehouse grain
    inventory_view = df.groupby([sku_col, wh_col]).agg(agg_dict).reset_index()
    
    # Clean up column names
    new_columns = []
    for col in inventory_view.columns:
        if col in [sku_col, wh_col]:
            new_columns.append(col)
        elif '_mean' in col:
            new_columns.append(col.replace('_mean', ''))
        elif '_sum' in col:
            new_columns.append(col.replace('_sum', ''))
        elif '_nunique' in col:
            new_columns.append(col.replace('_nunique', '_count'))
        else:
            new_columns.append(col)
    
    inventory_view.columns = new_columns
    
    # ==========================================
    # Step 4: Calculate recent period metrics
    # ==========================================
    
    # Recent sales (last 30, 60, 90 days)
    if sales_qty_col and date_col and sales_qty_col in df.columns:
        # Last 30 days
        sales_30 = df[df['days_ago'] <= 30].groupby([sku_col, wh_col])[sales_qty_col].sum().reset_index()
        sales_30.columns = [sku_col, wh_col, 'sales_last_30_days']
        
        # Last 60 days
        sales_60 = df[df['days_ago'] <= 60].groupby([sku_col, wh_col])[sales_qty_col].sum().reset_index()
        sales_60.columns = [sku_col, wh_col, 'sales_last_60_days']
        
        # Last 90 days
        sales_90 = df[df['days_ago'] <= 90].groupby([sku_col, wh_col])[sales_qty_col].sum().reset_index()
        sales_90.columns = [sku_col, wh_col, 'sales_last_90_days']
        
        # Merge with main view
        inventory_view = pd.merge(inventory_view, sales_30, on=[sku_col, wh_col], how='left')
        inventory_view = pd.merge(inventory_view, sales_60, on=[sku_col, wh_col], how='left')
        inventory_view = pd.merge(inventory_view, sales_90, on=[sku_col, wh_col], how='left')
    
    # Recent orders
    if orders_qty_col and date_col and orders_qty_col in df.columns:
        orders_30 = df[df['days_ago'] <= 30].groupby([sku_col, wh_col])[orders_qty_col].sum().reset_index()
        orders_30.columns = [sku_col, wh_col, 'orders_last_30_days']
        
        orders_60 = df[df['days_ago'] <= 60].groupby([sku_col, wh_col])[orders_qty_col].sum().reset_index()
        orders_60.columns = [sku_col, wh_col, 'orders_last_60_days']
        
        orders_90 = df[df['days_ago'] <= 90].groupby([sku_col, wh_col])[orders_qty_col].sum().reset_index()
        orders_90.columns = [sku_col, wh_col, 'orders_last_90_days']
        
        inventory_view = pd.merge(inventory_view, orders_30, on=[sku_col, wh_col], how='left')
        inventory_view = pd.merge(inventory_view, orders_60, on=[sku_col, wh_col], how='left')
        inventory_view = pd.merge(inventory_view, orders_90, on=[sku_col, wh_col], how='left')
    
    # ==========================================
    # Step 5: Calculate inventory health indicators
    # ==========================================
    
    # ✅ تصحیح: محاسبه days_of_inventory با استفاده از np.where روی مقادیر عددی
    if inv_col in inventory_view.columns and 'sales_last_30_days' in inventory_view.columns:
        # محاسبه میانگین فروش روزانه
        avg_daily_sales = inventory_view['sales_last_30_days'] / 30
        
        # محاسبه days_of_inventory با استفاده از np.where
        inventory_view['days_of_inventory'] = np.where(
            avg_daily_sales > 0,
            inventory_view[inv_col] / avg_daily_sales,
            0
        )
        
        # ✅ تصحیح: محاسبه inventory_turnover
        inventory_view['inventory_turnover'] = np.where(
            inventory_view[inv_col] > 0,
            inventory_view['sales_last_30_days'] / inventory_view[inv_col],
            0
        )
    
    # ==========================================
    # Step 6: Add warehouse information
    # ==========================================
    
    # If warehouse_name exists, add it
    if 'warehouse_name' in df.columns:
        warehouse_info = df[[wh_col, 'warehouse_name']].drop_duplicates()
        inventory_view = pd.merge(inventory_view, warehouse_info, on=wh_col, how='left')
    
    # ==========================================
    # Step 7: Add status flags (operational attention)
    # ==========================================
    
    # Out of stock with recent demand
    if inv_col in inventory_view.columns and 'sales_last_30_days' in inventory_view.columns:
        inventory_view['out_of_stock'] = np.where(
            (inventory_view[inv_col] == 0) & (inventory_view['sales_last_30_days'] > 0),
            1,
            0
        ).astype(int)
    
    # High stock (more than 90 days of inventory)
    if 'days_of_inventory' in inventory_view.columns:
        inventory_view['overstocked'] = np.where(
            inventory_view['days_of_inventory'] > 90,
            1,
            0
        ).astype(int)
    
    # Low stock (less than 7 days of inventory)
    if 'days_of_inventory' in inventory_view.columns:
        inventory_view['low_stock'] = np.where(
            (inventory_view['days_of_inventory'] < 7) & (inventory_view['days_of_inventory'] > 0),
            1,
            0
        ).astype(int)
    
    # Slow mover (no sales in last 90 days but has inventory)
    if 'sales_last_90_days' in inventory_view.columns and inv_col in inventory_view.columns:
        inventory_view['slow_mover'] = np.where(
            (inventory_view['sales_last_90_days'] == 0) & (inventory_view[inv_col] > 0),
            1,
            0
        ).astype(int)
    
    # ==========================================
    # Step 8: Add warehouse-level summaries
    # ==========================================
    
    # Add warehouse-level totals
    if inv_col in inventory_view.columns:
        wh_total_inv = inventory_view.groupby(wh_col)[inv_col].sum().reset_index()
        wh_total_inv.columns = [wh_col, 'warehouse_total_inventory']
        inventory_view = pd.merge(inventory_view, wh_total_inv, on=wh_col, how='left')
    
    if 'sales_last_30_days' in inventory_view.columns:
        wh_total_sales = inventory_view.groupby(wh_col)['sales_last_30_days'].sum().reset_index()
        wh_total_sales.columns = [wh_col, 'warehouse_total_sales_30d']
        inventory_view = pd.merge(inventory_view, wh_total_sales, on=wh_col, how='left')
    
    # ==========================================
    # Step 9: Sort and format
    # ==========================================
    
    # Sort by inventory value or sales
    if inv_col in inventory_view.columns and 'sales_last_30_days' in inventory_view.columns:
        inventory_view = inventory_view.sort_values(
            [inv_col, 'sales_last_30_days'], 
            ascending=[False, False]
        ).reset_index(drop=True)
    
    # Round numeric columns
    numeric_cols = inventory_view.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        inventory_view[col] = inventory_view[col].round(2)
    
    # ==========================================
    # Step 10: Log summary
    # ==========================================
    
    logger.info(f"\n  ✅ Inventory view created:")
    logger.info(f"    Rows: {len(inventory_view)}")
    logger.info(f"    Columns: {len(inventory_view.columns)}")
    
    if inv_col in inventory_view.columns:
        total_inventory = inventory_view[inv_col].sum()
        logger.info(f"    Total Inventory: {total_inventory:,.0f}")
    
    if 'sales_last_30_days' in inventory_view.columns:
        total_sales = inventory_view['sales_last_30_days'].sum()
        logger.info(f"    Total Sales (30 days): {total_sales:,.0f}")
    
    if 'out_of_stock' in inventory_view.columns:
        oos_count = inventory_view['out_of_stock'].sum()
        logger.info(f"    Out of Stock SKUs: {oos_count}")
    
    if 'overstocked' in inventory_view.columns:
        overstock_count = inventory_view['overstocked'].sum()
        logger.info(f"    Overstocked SKUs: {overstock_count}")
    
    if 'slow_mover' in inventory_view.columns:
        slow_mover_count = inventory_view['slow_mover'].sum()
        logger.info(f"    Slow Movers: {slow_mover_count}")
    
    return inventory_view

In [5]:
def create_warehouse_summary(inventory_view: pd.DataFrame) -> pd.DataFrame:
    """
    Create warehouse-level summary view
    
    Args:
        inventory_view: Inventory view dataframe
    
    Returns:
        Warehouse summary dataframe
    """
    logger.info("\n" + "="*60)
    logger.info("CREATING WAREHOUSE SUMMARY")
    logger.info("="*60)
    
    # Identify columns
    wh_col = None
    for col in inventory_view.columns:
        if 'warehouse' in col.lower() and 'id' in col.lower():
            wh_col = col
            break
    
    if wh_col is None:
        logger.error("  Could not find warehouse column")
        return None
    
    # Warehouse summary
    warehouse_summary = inventory_view.groupby(wh_col).agg({
        'sku': 'count',
        'on_hand_quantity': 'sum' if 'on_hand_quantity' in inventory_view.columns else None,
        'sales_last_30_days': 'sum' if 'sales_last_30_days' in inventory_view.columns else None,
        'orders_last_30_days': 'sum' if 'orders_last_30_days' in inventory_view.columns else None,
        'out_of_stock': 'sum' if 'out_of_stock' in inventory_view.columns else None,
        'overstocked': 'sum' if 'overstocked' in inventory_view.columns else None,
        'slow_mover': 'sum' if 'slow_mover' in inventory_view.columns else None,
    }).reset_index()
    
    # Rename columns
    warehouse_summary.columns = [
        col.replace('_sum', '').replace('_count', '_sku_count')
        if col != wh_col else col
        for col in warehouse_summary.columns
    ]
    
    # Add warehouse_name if available
    if 'warehouse_name' in inventory_view.columns:
        wh_names = inventory_view[[wh_col, 'warehouse_name']].drop_duplicates()
        warehouse_summary = pd.merge(warehouse_summary, wh_names, on=wh_col, how='left')
    
    logger.info(f"  ✅ Warehouse summary created: {len(warehouse_summary)} rows")
    
    return warehouse_summary

In [6]:
def create_sku_performance_summary(inventory_view: pd.DataFrame) -> pd.DataFrame:
    """
    Create SKU performance summary across all warehouses
    
    Args:
        inventory_view: Inventory view dataframe
    
    Returns:
        SKU performance dataframe
    """
    logger.info("\n" + "="*60)
    logger.info("CREATING SKU PERFORMANCE SUMMARY")
    logger.info("="*60)
    
    # Identify SKU column
    sku_col = None
    for col in inventory_view.columns:
        if col.lower() == 'sku':
            sku_col = col
            break
    
    if sku_col is None:
        logger.error("  Could not find SKU column")
        return None
    
    # SKU summary across all warehouses
    sku_summary = inventory_view.groupby(sku_col).agg({
        'on_hand_quantity': 'sum' if 'on_hand_quantity' in inventory_view.columns else None,
        'sales_last_30_days': 'sum' if 'sales_last_30_days' in inventory_view.columns else None,
        'orders_last_30_days': 'sum' if 'orders_last_30_days' in inventory_view.columns else None,
        'warehouse_id': 'nunique',
        'out_of_stock': 'sum' if 'out_of_stock' in inventory_view.columns else None,
    }).reset_index()
    
    # Rename columns
    sku_summary.columns = [
        'sku' if col == sku_col else
        col.replace('_sum', '').replace('_nunique', '_warehouse_count')
        if col != sku_col else col
        for col in sku_summary.columns
    ]
    
    # Add classification
    if 'sales_last_30_days' in sku_summary.columns:
        # Classify by sales volume
        sku_summary['performance_tier'] = pd.cut(
            sku_summary['sales_last_30_days'],
            bins=[-1, 0, 10, 100, float('inf')],
            labels=['No Sales', 'Low', 'Medium', 'High']
        )
    
    if 'on_hand_quantity' in sku_summary.columns:
        # Stock status across all warehouses
        sku_summary['total_stock_level'] = pd.cut(
            sku_summary['on_hand_quantity'],
            bins=[-1, 0, 50, 200, float('inf')],
            labels=['Out of Stock', 'Low', 'Medium', 'High']
        )
    
    logger.info(f"  ✅ SKU summary created: {len(sku_summary)} rows")
    
    return sku_summary

In [7]:
def prepare_visualization_data(inventory_view: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Prepare data for visualizations
    
    Returns:
        Dictionary of dataframes for different visualizations
    """
    logger.info("\n" + "="*60)
    logger.info("PREPARING VISUALIZATION DATA")
    logger.info("="*60)
    
    viz_data = {}
    
    # 1. Top selling SKUs by warehouse
    if 'sku' in inventory_view.columns and 'warehouse_id' in inventory_view.columns:
        if 'sales_last_30_days' in inventory_view.columns:
            top_skus = inventory_view.nlargest(20, 'sales_last_30_days')[
                ['sku', 'warehouse_id', 'warehouse_name', 'on_hand_quantity', 
                 'sales_last_30_days', 'days_of_inventory']
            ]
            viz_data['top_skus'] = top_skus
            logger.info(f"  - Top 20 SKUs by sales")
    
    # 2. Warehouse inventory distribution
    if 'warehouse_id' in inventory_view.columns and 'on_hand_quantity' in inventory_view.columns:
        wh_inv = inventory_view.groupby('warehouse_id')['on_hand_quantity'].sum().reset_index()
        wh_inv.columns = ['warehouse_id', 'total_inventory']
        viz_data['warehouse_inventory'] = wh_inv
        logger.info(f"  - Warehouse inventory distribution")
    
    # 3. Inventory health status
    health_cols = ['out_of_stock', 'overstocked', 'slow_mover', 'low_stock']
    health_data = {}
    for col in health_cols:
        if col in inventory_view.columns:
            health_data[col] = inventory_view[col].sum()
    
    if health_data:
        health_df = pd.DataFrame([
            {'status': k.replace('_', ' ').title(), 'count': v}
            for k, v in health_data.items() if v > 0
        ])
        viz_data['inventory_health'] = health_df
        logger.info(f"  - Inventory health summary")
    
    # 4. SKU concentration (Pareto analysis)
    if 'sales_last_30_days' in inventory_view.columns:
        sales_sorted = inventory_view.sort_values('sales_last_30_days', ascending=False).copy()
        
        # ✅ تصحیح: محاسبه درصد SKU با استفاده از numpy
        n_rows = len(sales_sorted)
        sales_sorted['sku_percent'] = np.arange(1, n_rows + 1) / n_rows * 100
        
        # محاسبه فروش تجمعی
        sales_sorted['cumulative_sales'] = sales_sorted['sales_last_30_days'].cumsum()
        total_sales = sales_sorted['sales_last_30_days'].sum()
        
        # ✅ تصحیح: جلوگیری از تقسیم بر صفر
        if total_sales > 0:
            sales_sorted['cumulative_percent'] = (sales_sorted['cumulative_sales'] / total_sales) * 100
        else:
            sales_sorted['cumulative_percent'] = 0
        
        # نگه داشتن ستون‌های مورد نیاز
        pareto_cols = ['sku', 'sales_last_30_days', 'cumulative_percent', 'sku_percent']
        pareto_cols = [col for col in pareto_cols if col in sales_sorted.columns]
        viz_data['pareto_analysis'] = sales_sorted[pareto_cols]
        
        # ✅ اضافه کردن نقطه Pareto (80% فروش)
        if total_sales > 0:
            # پیدا کردن نقطه‌ای که 80% فروش را پوشش می‌دهد
            pareto_80 = sales_sorted[sales_sorted['cumulative_percent'] <= 80]
            if not pareto_80.empty:
                sku_count_80 = len(pareto_80)
                sku_percent_80 = (sku_count_80 / n_rows) * 100
                logger.info(f"  - Pareto analysis: {sku_count_80} SKUs ({sku_percent_80:.1f}%) account for 80% of sales")
        
        logger.info(f"  - Pareto analysis prepared")
    
    return viz_data

In [8]:
def save_analytical_outputs(
    inventory_view: pd.DataFrame,
    warehouse_summary: pd.DataFrame,
    sku_summary: pd.DataFrame,
    viz_data: Dict[str, pd.DataFrame],
    output_dir: Path
) -> None:
    """
    Save all analytical outputs
    
    Args:
        inventory_view: Main inventory view
        warehouse_summary: Warehouse summary
        sku_summary: SKU summary
        viz_data: Visualization data
        output_dir: Output directory
    """
    logger.info("\n" + "="*60)
    logger.info("SAVING ANALYTICAL OUTPUTS")
    logger.info("="*60)
    
    # Create analytics directory
    analytics_dir = output_dir / 'analytics'
    analytics_dir.mkdir(parents=True, exist_ok=True)
    
    # Save main inventory view
    inv_path = analytics_dir / 'inventory_view.csv'
    inventory_view.to_csv(inv_path, index=False)
    logger.info(f"  ✅ Saved inventory view: {inv_path}")
    
    # Save warehouse summary
    if warehouse_summary is not None:
        wh_path = analytics_dir / 'warehouse_summary.csv'
        warehouse_summary.to_csv(wh_path, index=False)
        logger.info(f"  ✅ Saved warehouse summary: {wh_path}")
    
    # Save SKU summary
    if sku_summary is not None:
        sku_path = analytics_dir / 'sku_performance_summary.csv'
        sku_summary.to_csv(sku_path, index=False)
        logger.info(f"  ✅ Saved SKU summary: {sku_path}")
    
    # Save visualization data
    for name, data in viz_data.items():
        if not data.empty:
            path = analytics_dir / f'viz_{name}.csv'
            data.to_csv(path, index=False)
            logger.info(f"  ✅ Saved viz data: {path}")
    
    # Create a simple summary JSON
    summary = {
        'total_skus': len(inventory_view),
        'total_warehouses': inventory_view['warehouse_id'].nunique() if 'warehouse_id' in inventory_view.columns else 0,
        'total_inventory': inventory_view['on_hand_quantity'].sum() if 'on_hand_quantity' in inventory_view.columns else 0,
        'total_sales_30d': inventory_view['sales_last_30_days'].sum() if 'sales_last_30_days' in inventory_view.columns else 0,
        'out_of_stock_count': inventory_view['out_of_stock'].sum() if 'out_of_stock' in inventory_view.columns else 0,
        'overstocked_count': inventory_view['overstocked'].sum() if 'overstocked' in inventory_view.columns else 0,
        'slow_mover_count': inventory_view['slow_mover'].sum() if 'slow_mover' in inventory_view.columns else 0,
    }
    
    summary_path = analytics_dir / 'analytics_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    logger.info(f"  ✅ Saved analytics summary: {summary_path}")
    
    logger.info(f"\n  ✅ All analytical outputs saved to: {analytics_dir}")

In [9]:
def run_q5_analytics(processed_dir: Path) -> Dict[str, pd.DataFrame]:
    """
    Run Q5 analytics - Create single trusted inventory view
    
    Args:
        processed_dir: Directory containing processed data
    
    Returns:
        Dictionary with all analytical dataframes
    """
    logger.info("="*60)
    logger.info("STARTING Q5 - SINGLE TRUSTED INVENTORY VIEW")
    logger.info("="*60)
    
    # Load integrated data
    integrated_dir = processed_dir / 'integrated'
    integrated_df = load_integrated_data(integrated_dir)
    
    if integrated_df is None:
        logger.error("  Could not load integrated data")
        return None
    
    # Create inventory view
    inventory_view = create_inventory_view(integrated_df)
    
    if inventory_view is None:
        logger.error("  Could not create inventory view")
        return None
    
    # Create warehouse summary
    warehouse_summary = create_warehouse_summary(inventory_view)
    
    # Create SKU performance summary
    sku_summary = create_sku_performance_summary(inventory_view)
    
    # Prepare visualization data
    viz_data = prepare_visualization_data(inventory_view)
    
    # Save all outputs
    save_analytical_outputs(
        inventory_view=inventory_view,
        warehouse_summary=warehouse_summary,
        sku_summary=sku_summary,
        viz_data=viz_data,
        output_dir=processed_dir
    )
    
    # Return all data
    results = {
        'inventory_view': inventory_view,
        'warehouse_summary': warehouse_summary,
        'sku_summary': sku_summary,
        'viz_data': viz_data
    }
    
    logger.info("\n" + "="*60)
    logger.info("✅ Q5 - SINGLE TRUSTED INVENTORY VIEW COMPLETED")
    logger.info("="*60)
    
    return results

##### Q6 - Identify warehouses/SKUs that may need attention

In [10]:
def load_inventory_view(analytics_dir: Path) -> pd.DataFrame:
    """
    Load the inventory view created in Q5
    
    Args:
        analytics_dir: Path to analytics directory
    
    Returns:
        Inventory view dataframe
    """
    logger.info(f"Loading inventory view from: {analytics_dir}")
    
    filepath = analytics_dir / 'inventory_view.csv'
    
    if filepath.exists():
        df = pd.read_csv(filepath)
        logger.info(f"  - Loaded inventory view: {len(df)} rows, {len(df.columns)} columns")
        return df
    else:
        logger.error(f"  - Inventory view not found: {filepath}")
        return None

In [11]:
def identify_out_of_stock_with_demand(
    inventory_view: pd.DataFrame,
    demand_days: int = 30
) -> pd.DataFrame:
    """
    Rule 1: Out of stock with recent demand
    
    Identifies SKUs where:
    - Current inventory = 0
    - There was demand (sales or orders) in the last N days
    
    Args:
        inventory_view: Inventory view dataframe
        demand_days: Number of days to look back for demand
    
    Returns:
        DataFrame with out-of-stock alerts
    """
    logger.info("\n" + "="*60)
    logger.info("RULE 1: OUT OF STOCK WITH DEMAND")
    logger.info("="*60)
    
    # Identify relevant columns
    inv_col = None
    for col in inventory_view.columns:
        if 'on_hand' in col.lower():
            inv_col = col
            break
    
    sales_col = None
    for col in inventory_view.columns:
        if f'sales_last_{demand_days}_days' in col:
            sales_col = col
            break
    
    orders_col = None
    for col in inventory_view.columns:
        if f'orders_last_{demand_days}_days' in col:
            orders_col = col
            break
    
    if inv_col is None:
        logger.error("  Could not find inventory column")
        return pd.DataFrame()
    
    # Create alert dataframe
    alerts = inventory_view.copy()
    
    # Condition: out of stock AND (sales > 0 OR orders > 0)
    condition = (alerts[inv_col] == 0)
    
    if sales_col:
        condition = condition & (alerts[sales_col] > 0)
    
    if orders_col:
        condition = condition & (alerts[orders_col] > 0)
    
    alerts = alerts[condition].copy()
    
    # Add severity and recommendation
    if not alerts.empty:
        alerts['alert_type'] = 'out_of_stock_with_demand'
        alerts['severity'] = 'high'
        
        # Calculate urgency based on demand volume
        if sales_col:
            alerts['urgent_score'] = alerts[sales_col] / alerts[sales_col].max() * 100
            alerts['recommendation'] = alerts['urgent_score'].apply(
                lambda x: 'URGENT: Replenish immediately' if x > 70 else
                          'Replenish as soon as possible' if x > 40 else
                          'Schedule replenishment'
            )
        else:
            alerts['recommendation'] = 'Replenish as soon as possible'
        
        logger.info(f"  ✅ Found {len(alerts)} out-of-stock SKUs with demand")
        if sales_col:
            logger.info(f"    Total lost sales: {alerts[sales_col].sum():,.0f}")
    else:
        logger.info("  ✅ No out-of-stock SKUs with demand found")
    
    return alerts

In [12]:
def identify_overstocked(
    inventory_view: pd.DataFrame,
    max_days_inventory: int = 90
) -> pd.DataFrame:
    """
    Rule 2: Unusually high stock (overstocked)
    
    Identifies SKUs where:
    - Days of inventory > threshold (e.g., 90 days)
    - Inventory value is high
    
    Args:
        inventory_view: Inventory view dataframe
        max_days_inventory: Maximum days of inventory before flagged
    
    Returns:
        DataFrame with overstock alerts
    """
    logger.info("\n" + "="*60)
    logger.info(f"RULE 2: OVERSTOCKED (> {max_days_inventory} days)")
    logger.info("="*60)
    
    # Identify relevant columns
    inv_col = None
    for col in inventory_view.columns:
        if 'on_hand' in col.lower():
            inv_col = col
            break
    
    days_col = None
    for col in inventory_view.columns:
        if 'days_of_inventory' in col.lower():
            days_col = col
            break
    
    sales_col = None
    for col in inventory_view.columns:
        if 'sales_last_30_days' in col:
            sales_col = col
            break
    
    if inv_col is None or days_col is None:
        logger.error("  Could not find inventory or days column")
        return pd.DataFrame()
    
    # Create alert dataframe
    alerts = inventory_view.copy()
    
    # Condition: days of inventory > threshold
    alerts = alerts[alerts[days_col] > max_days_inventory].copy()
    
    if not alerts.empty:
        alerts['alert_type'] = 'overstocked'
        alerts['severity'] = 'medium'
        
        # Calculate excess inventory
        if sales_col:
            avg_daily_sales = alerts[sales_col] / 30
            alerts['excess_days'] = alerts[days_col] - max_days_inventory
            alerts['excess_inventory'] = alerts['excess_days'] * avg_daily_sales
        
        # Recommendation
        alerts['recommendation'] = alerts[days_col].apply(
            lambda x: 'URGENT: Move inventory' if x > 180 else
                      'Consider promotion or transfer' if x > 120 else
                      'Monitor and reduce orders'
        )
        
        # Financial impact
        alerts['priority_score'] = alerts[inv_col] / alerts[inv_col].max() * 100
        
        logger.info(f"  ✅ Found {len(alerts)} overstocked SKUs")
        logger.info(f"    Total excess inventory value: {alerts[inv_col].sum():,.0f}")
        logger.info(f"    Average days of inventory: {alerts[days_col].mean():.0f}")
    else:
        logger.info("  ✅ No overstocked SKUs found")
    
    return alerts

In [13]:
def identify_orders_without_stock(
    inventory_view: pd.DataFrame,
    threshold_days: int = 30
) -> pd.DataFrame:
    """
    Rule 3: Orders without sufficient on-hand quantity
    
    Identifies SKUs where:
    - There are pending orders
    - Current inventory < open orders
    - Or inventory is too low to fulfill orders
    
    Args:
        inventory_view: Inventory view dataframe
        threshold_days: Lookback period for orders
    
    Returns:
        DataFrame with stock-out risk alerts
    """
    logger.info("\n" + "="*60)
    logger.info("RULE 3: ORDERS WITHOUT SUFFICIENT STOCK")
    logger.info("="*60)
    
    # Identify relevant columns
    inv_col = None
    for col in inventory_view.columns:
        if 'on_hand' in col.lower():
            inv_col = col
            break
    
    orders_col = None
    for col in inventory_view.columns:
        if f'orders_last_{threshold_days}_days' in col:
            orders_col = col
            break
    
    if inv_col is None or orders_col is None:
        logger.error("  Could not find inventory or orders column")
        return pd.DataFrame()
    
    # Create alert dataframe
    alerts = inventory_view.copy()
    
    # ✅ محاسبه stock_coverage قبل از استفاده در شرط
    alerts['stock_coverage'] = alerts[inv_col] - alerts[orders_col]
    alerts['insufficient_stock'] = alerts['stock_coverage'] < 0
    
    # Condition: orders > 0 AND insufficient stock
    condition = (alerts[orders_col] > 0) & alerts['insufficient_stock']
    alerts = alerts[condition].copy()
    
    if not alerts.empty:
        alerts['alert_type'] = 'orders_without_stock'
        alerts['severity'] = 'critical'
        
        # Calculate stock deficit
        alerts['stock_deficit'] = -alerts['stock_coverage']
        
        # Recommendation
        alerts['recommendation'] = alerts['stock_deficit'].apply(
            lambda x: 'CRITICAL: Immediate replenishment needed' if x > 100 else
                      'URGENT: Expedite shipment' if x > 50 else
                      'Consider partial fulfillment'
        )
        
        # Priority based on order volume and deficit
        alerts['priority_score'] = (alerts[orders_col] * alerts['stock_deficit']) / \
                                   (alerts[orders_col] * alerts['stock_deficit']).max() * 100
        
        logger.info(f"  ✅ Found {len(alerts)} SKUs with insufficient stock for orders")
        logger.info(f"    Total order backlog: {alerts[orders_col].sum():,.0f}")
        logger.info(f"    Total stock deficit: {alerts['stock_deficit'].sum():,.0f}")
    else:
        logger.info("  ✅ No SKUs with insufficient stock for orders found")
    
    return alerts

In [14]:
def identify_slow_movers(
    inventory_view: pd.DataFrame,
    lookback_days: int = 90
) -> pd.DataFrame:
    """
    Rule 4: Slow movers (no recent sales but high inventory)
    
    Identifies SKUs where:
    - No sales in last N days
    - Inventory is significant (blocking capital)
    
    Args:
        inventory_view: Inventory view dataframe
        lookback_days: Number of days to check for sales
    
    Returns:
        DataFrame with slow mover alerts
    """
    logger.info("\n" + "="*60)
    logger.info(f"RULE 4: SLOW MOVERS (No sales in {lookback_days} days)")
    logger.info("="*60)
    
    # Identify relevant columns
    inv_col = None
    for col in inventory_view.columns:
        if 'on_hand' in col.lower():
            inv_col = col
            break
    
    sales_col = None
    for col in inventory_view.columns:
        if f'sales_last_{lookback_days}_days' in col:
            sales_col = col
            break
    
    if inv_col is None or sales_col is None:
        logger.error("  Could not find inventory or sales column")
        return pd.DataFrame()
    
    # Create alert dataframe
    alerts = inventory_view.copy()
    
    # Condition: no sales AND inventory > 0
    condition = (alerts[sales_col] == 0) & (alerts[inv_col] > 0)
    alerts = alerts[condition].copy()
    
    if not alerts.empty:
        alerts['alert_type'] = 'slow_mover'
        alerts['severity'] = 'medium'
        
        # Category based on inventory value
        alerts['category'] = alerts[inv_col].apply(
            lambda x: 'High value slow mover' if x > 1000 else
                      'Medium value slow mover' if x > 100 else
                      'Low value slow mover'
        )
        
        # Recommendation
        alerts['recommendation'] = alerts[inv_col].apply(
            lambda x: 'URGENT: Review and markdown' if x > 1000 else
                      'Consider consolidation' if x > 100 else
                      'Monitor and review'
        )
        
        # Capital blocked
        alerts['capital_blocked'] = alerts[inv_col] * 1  # Assuming average cost per unit
        
        logger.info(f"  ✅ Found {len(alerts)} slow-moving SKUs")
        logger.info(f"    Total capital blocked: ${alerts['capital_blocked'].sum():,.0f}")
        logger.info(f"    Total inventory: {alerts[inv_col].sum():,.0f} units")
    else:
        logger.info("  ✅ No slow-moving SKUs found")
    
    return alerts

In [15]:
def identify_warehouse_imbalance(
    inventory_view: pd.DataFrame
) -> pd.DataFrame:
    """
    Rule 5: Warehouse imbalance (some warehouses have too much, others too little)
    
    Identifies SKUs that are overstocked in some warehouses and
    understocked in others
    
    Args:
        inventory_view: Inventory view dataframe
    
    Returns:
        DataFrame with warehouse imbalance alerts
    """
    logger.info("\n" + "="*60)
    logger.info("RULE 5: WAREHOUSE IMBALANCE")
    logger.info("="*60)
    
    # Identify relevant columns
    inv_col = None
    for col in inventory_view.columns:
        if 'on_hand' in col.lower():
            inv_col = col
            break
    
    sku_col = None
    for col in inventory_view.columns:
        if col.lower() == 'sku':
            sku_col = col
            break
    
    wh_col = None
    for col in inventory_view.columns:
        if col.lower() == 'warehouse_id':
            wh_col = col
            break
    
    if inv_col is None or sku_col is None or wh_col is None:
        logger.error("  Could not find required columns")
        return pd.DataFrame()
    
    # ✅ مطمئن شوید که ستون‌ها از نوع عددی هستند
    inventory_view[inv_col] = pd.to_numeric(inventory_view[inv_col], errors='coerce').fillna(0)
    
    # Calculate SKU-level statistics across warehouses
    sku_stats = inventory_view.groupby(sku_col).agg({
        inv_col: ['mean', 'std', 'sum', 'count']
    }).reset_index()
    
    sku_stats.columns = [sku_col, 'avg_inventory', 'std_inventory', 'total_inventory', 'warehouse_count']
    
    # ✅ جلوگیری از تقسیم بر صفر
    sku_stats['cv'] = np.where(
        sku_stats['avg_inventory'] > 0,
        sku_stats['std_inventory'] / sku_stats['avg_inventory'],
        0
    )
    
    # Identify SKUs with high imbalance (CV > 0.8 and at least 2 warehouses)
    imbalanced_skus = sku_stats[
        (sku_stats['cv'] > 0.8) & 
        (sku_stats['warehouse_count'] >= 2)
    ].copy()
    
    alerts = pd.DataFrame()
    
    if not imbalanced_skus.empty:
        for _, sku_row in imbalanced_skus.iterrows():
            sku = sku_row[sku_col]
            
            # Get warehouse distribution for this SKU
            sku_wh_data = inventory_view[inventory_view[sku_col] == sku].copy()
            
            if len(sku_wh_data) > 1:
                avg_inv = sku_row['avg_inventory']
                
                if avg_inv > 0:
                    sku_wh_data['deviation'] = sku_wh_data[inv_col] - avg_inv
                    sku_wh_data['relative_deviation'] = (sku_wh_data['deviation'] / avg_inv) * 100
                    
                    # Overstocked (positive deviation > 50%)
                    overstocked = sku_wh_data[sku_wh_data['relative_deviation'] > 50].copy()
                    if not overstocked.empty:
                        overstocked['alert_type'] = 'warehouse_overstocked'
                        overstocked['severity'] = 'medium'
                        overstocked['recommendation'] = 'Consider transferring to understocked warehouses'
                        alerts = pd.concat([alerts, overstocked], ignore_index=True)
                    
                    # Understocked (negative deviation < -50%)
                    understocked = sku_wh_data[sku_wh_data['relative_deviation'] < -50].copy()
                    if not understocked.empty:
                        understocked['alert_type'] = 'warehouse_understocked'
                        understocked['severity'] = 'high'
                        understocked['recommendation'] = 'URGENT: Receive transfer from overstocked warehouses'
                        alerts = pd.concat([alerts, understocked], ignore_index=True)
        
        if not alerts.empty:
            # ✅ اضافه کردن ستون‌های مورد نیاز
            if 'sku' not in alerts.columns and sku_col in alerts.columns:
                alerts = alerts.rename(columns={sku_col: 'sku'})
            if 'warehouse_id' not in alerts.columns and wh_col in alerts.columns:
                alerts = alerts.rename(columns={wh_col: 'warehouse_id'})
            
            logger.info(f"  ✅ Found {len(alerts)} warehouse imbalance alerts")
            logger.info(f"    Affected SKUs: {alerts['sku'].nunique() if 'sku' in alerts.columns else 0}")
            if 'alert_type' in alerts.columns:
                overstocked_count = len(alerts[alerts['alert_type'] == 'warehouse_overstocked'])
                understocked_count = len(alerts[alerts['alert_type'] == 'warehouse_understocked'])
                logger.info(f"    Overstocked alerts: {overstocked_count}")
                logger.info(f"    Understocked alerts: {understocked_count}")
        else:
            logger.info("  ✅ No warehouse imbalance issues found")
    else:
        logger.info("  ✅ No SKUs with significant warehouse imbalance found")
    
    return alerts

In [16]:
def generate_attention_alerts(inventory_view: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Generate all attention alerts for Q6
    
    Args:
        inventory_view: Inventory view dataframe
    
    Returns:
        Dictionary of alert dataframes
    """
    logger.info("\n" + "="*60)
    logger.info("Q6: GENERATING ATTENTION ALERTS")
    logger.info("="*60)
    
    alerts = {}
    
    # Rule 1: Out of stock with demand
    alerts['out_of_stock'] = identify_out_of_stock_with_demand(
        inventory_view, demand_days=30
    )
    
    # Rule 2: Overstocked
    alerts['overstocked'] = identify_overstocked(
        inventory_view, max_days_inventory=90
    )
    
    # Rule 3: Orders without sufficient stock
    alerts['orders_without_stock'] = identify_orders_without_stock(
        inventory_view, threshold_days=30
    )
    
    # Rule 4: Slow movers
    alerts['slow_movers'] = identify_slow_movers(
        inventory_view, lookback_days=90
    )
    
    # Rule 5: Warehouse imbalance
    alerts['warehouse_imbalance'] = identify_warehouse_imbalance(
        inventory_view
    )
    
    # Combined summary
    logger.info("\n" + "="*60)
    logger.info("Q6: ATTENTION ALERTS SUMMARY")
    logger.info("="*60)
    
    total_alerts = 0
    for name, df in alerts.items():
        if df is not None and not df.empty:
            total_alerts += len(df)
            logger.info(f"  {name.replace('_', ' ').title()}: {len(df)} alerts")
    
    logger.info(f"\n  ✅ Total alerts generated: {total_alerts}")
    
    return alerts

In [17]:
def save_q6_outputs(
    alerts: Dict[str, pd.DataFrame],
    output_dir: Path
) -> None:
    """
    Save Q6 attention alerts
    
    Args:
        alerts: Dictionary of alert dataframes
        output_dir: Output directory
    """
    logger.info("\n" + "="*60)
    logger.info("SAVING Q6 ATTENTION ALERTS")
    logger.info("="*60)
    
    # Create Q6 directory
    q6_dir = output_dir / 'analytics' / 'attention_alerts'
    q6_dir.mkdir(parents=True, exist_ok=True)
    
    # Save each alert type
    for name, df in alerts.items():
        if df is not None and not df.empty:
            filepath = q6_dir / f'{name}_alerts.csv'
            df.to_csv(filepath, index=False)
            logger.info(f"  ✅ Saved {name}: {filepath}")
        else:
            logger.info(f"  - No alerts for {name}")
    
    # Create combined alerts file
    combined = pd.DataFrame()
    for name, df in alerts.items():
        if df is not None and not df.empty:
            df_copy = df.copy()
            df_copy['alert_source'] = name
            combined = pd.concat([combined, df_copy], ignore_index=True)
    
    if not combined.empty:
        # Sort by severity
        severity_order = {'critical': 0, 'high': 1, 'medium': 2, 'low': 3}
        combined['severity_rank'] = combined['severity'].map(severity_order).fillna(4)
        combined = combined.sort_values('severity_rank').drop(columns=['severity_rank'])
        
        # Save combined
        combined_path = q6_dir / 'all_alerts_combined.csv'
        combined.to_csv(combined_path, index=False)
        logger.info(f"  ✅ Saved combined alerts: {combined_path}")
        
        # Create summary statistics
        summary = {
            'total_alerts': len(combined),
            'by_type': combined['alert_type'].value_counts().to_dict() if 'alert_type' in combined.columns else {},
            'by_severity': combined['severity'].value_counts().to_dict() if 'severity' in combined.columns else {},
            'unique_skus': combined['sku'].nunique() if 'sku' in combined.columns else 0,
            'unique_warehouses': combined['warehouse_id'].nunique() if 'warehouse_id' in combined.columns else 0,
            'generated_at': datetime.now().isoformat()
        }
        
        summary_path = q6_dir / 'alerts_summary.json'
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        logger.info(f"  ✅ Saved alerts summary: {summary_path}")

In [18]:
def run_q6_analysis(processed_dir: Path) -> Dict[str, pd.DataFrame]:
    """
    Run Q6 analysis - Identify warehouses/SKUs that need attention
    
    Args:
        processed_dir: Directory containing processed data
    
    Returns:
        Dictionary with all alert dataframes
    """
    logger.info("="*60)
    logger.info("STARTING Q6 - ATTENTION ALERTS")
    logger.info("="*60)
    
    # Load inventory view from Q5
    analytics_dir = processed_dir / 'analytics'
    inventory_view = load_inventory_view(analytics_dir)
    
    if inventory_view is None:
        logger.error("  Could not load inventory view")
        return None
    
    # Generate attention alerts
    alerts = generate_attention_alerts(inventory_view)
    
    # Save outputs
    save_q6_outputs(alerts, processed_dir)
    
    logger.info("\n" + "="*60)
    logger.info("✅ Q6 - ATTENTION ALERTS COMPLETED")
    logger.info("="*60)
    
    return alerts